# 01 — Environment and Project Check

Run this first after opening the repo.

This notebook checks:

1. Python and core packages
2. CUDA / CPU availability
3. Project root and `src.utils.paths`
4. Raw dataset folders
5. Processed manifest files
6. Flat processed image folder
7. Split ID files
8. Paper/cache folders

Expected order:

```text
1. env_check_updated.ipynb
2. data_setup_updated.ipynb
3. eda_updated.ipynb
```


In [33]:
from pathlib import Path
import sys
import importlib
import json
import platform

# Find project root.
cwd = Path.cwd()
if (cwd / "data").exists() and (cwd / "src").exists():
    ROOT = cwd
else:
    ROOT = cwd.parent

sys.path.insert(0, str(ROOT))

print("ROOT:", ROOT)
print("Python:", sys.version)
print("Platform:", platform.platform())

def check(label, condition, detail=""):
    mark = "✓" if condition else "✗"
    print(f"{mark} {label}" + (f" — {detail}" if detail else ""))
    return bool(condition)


ROOT: e:\eve-rag\notebooks
Python: 3.13.13 (tags/v3.13.13:01104ce, Apr  7 2026, 19:25:48) [MSC v.1944 64 bit (AMD64)]
Platform: Windows-11-10.0.26200-SP0


## 1. Python version


In [34]:
v = sys.version_info
check("Python >= 3.10", v.major == 3 and v.minor >= 10, f"found {v.major}.{v.minor}.{v.micro}")


✓ Python >= 3.10 — found 3.13.13


True

## 2. Core packages


In [35]:
packages = [
    ("numpy", "numpy"),
    ("pandas", "pandas"),
    ("matplotlib", "matplotlib"),
    ("sklearn", "scikit-learn"),
    ("PIL", "Pillow"),
    ("tqdm", "tqdm"),
    ("torch", "torch"),
    ("transformers", "transformers"),
    ("spacy", "spacy"),
]

missing = []
for import_name, pip_name in packages:
    try:
        mod = importlib.import_module(import_name)
        version = getattr(mod, "__version__", "ok")
        check(f"package {pip_name}", True, str(version))
    except Exception as e:
        check(f"package {pip_name}", False, "missing")
        missing.append(pip_name)

if missing:
    print("\nInstall missing packages with:")
    print("pip install " + " ".join(missing))
else:
    print("\nAll core packages available.")


✓ package numpy — 2.4.4
✓ package pandas — 3.0.2
✓ package matplotlib — 3.10.8
✓ package scikit-learn — 1.8.0
✓ package Pillow — 12.2.0
✓ package tqdm — 4.67.3
✓ package torch — 2.6.0+cu124
✓ package transformers — 5.5.3
✓ package spacy — 3.8.14

All core packages available.


## 3. PyTorch device


In [36]:
try:
    import torch
    print("PyTorch:", torch.__version__)
    if torch.cuda.is_available():
        name = torch.cuda.get_device_name(0)
        mem = torch.cuda.get_device_properties(0).total_memory / 1e9
        check("CUDA available", True, f"{name}, {mem:.1f} GB")
        DEVICE = "cuda"
    else:
        check("CUDA available", False, "CPU-only is okay for setup/EDA; training may need GPU")
        DEVICE = "cpu"
    x = torch.ones(2, 2).to(DEVICE)
    check("tensor smoke test", True, str(x.device))
except Exception as e:
    check("PyTorch check", False, repr(e))


PyTorch: 2.6.0+cu124
✓ CUDA available — NVIDIA GeForce RTX 4060 Ti, 8.6 GB
✓ tensor smoke test — cuda:0


## 4. Import project paths


In [37]:
from pathlib import Path
import sys

def find_repo_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for p in [start, *start.parents]:
        if (p / "data").exists() and (p / "src").exists():
            return p
    raise RuntimeError("Could not find repo root. Start Jupyter from E:/eve-rag or place notebook inside the repo.")

ROOT = find_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("Detected repo root:", ROOT)
print("src exists:", (ROOT / "src").exists())

try:
    from src.utils.paths import (
        ROOT as PATH_ROOT,
        DATA,
        RAW,
        PROCESSED,
        MANIFESTS,
        PROCESSED_IMAGES,
        REPORTS,
        SPLITS,
        TRAIN_MANIFEST,
        VAL_MANIFEST,
        TEST_MANIFEST,
        VERITE_TEST_MANIFEST,
        MMFAKEBENCH_VAL_MANIFEST,
        MMFAKEBENCH_TEST_MANIFEST,
        ALL_MANIFEST,
    )

    check("src.utils.paths import", True)
    print("PATH_ROOT:", PATH_ROOT)

except Exception as e:
    check("src.utils.paths import", False, repr(e))
    raise

Detected repo root: e:\eve-rag
src exists: True
✓ src.utils.paths import
PATH_ROOT: E:\eve-rag


## 5. Raw dataset folders


In [38]:
from pathlib import Path
import sys

def find_repo_root(start=None):
    start = Path.cwd() if start is None else Path(start)

    for p in [start, *start.parents]:
        if (p / "data").exists() and (p / "src").exists():
            return p

    raise RuntimeError(
        "Could not find repo root. Start Jupyter from E:/eve-rag or place notebook inside the repo."
    )

ROOT = find_repo_root()

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("Detected repo root:", ROOT)
print("src exists:", (ROOT / "src").exists())

try:
    from src.utils.paths import (
        ROOT as PATH_ROOT,

        DATA,
        RAW,
        PROCESSED,
        MANIFESTS,
        PROCESSED_IMAGES,
        REPORTS,
        SPLITS,
        EXTERNAL,
        PRETRAINED,

        NEWSCLIPPINGS_RAW,
        MMFAKEBENCH_RAW,
        VERITE_RAW,
        VERITE_CANON,
        VERITE_CSV,
        VERITE_ARTICLES,
        VERITE_IMAGES,

        TRAIN_MANIFEST,
        VAL_MANIFEST,
        TEST_MANIFEST,
        VERITE_TEST_MANIFEST,
        MMFAKEBENCH_VAL_MANIFEST,
        MMFAKEBENCH_TEST_MANIFEST,
        ALL_MANIFEST,

        MANIFEST_SUMMARY,

        CACHE,
        STREAM_A_CACHE,
        STREAM_B_CACHE,
        STREAM_C_CACHE,
        FUSION_CACHE,

        PAPER,
        FIGURES,
        TABLES,
    )

    check("src.utils.paths import", True)
    print("PATH_ROOT:", PATH_ROOT)

except Exception as e:
    check("src.utils.paths import", False, repr(e))
    raise

Detected repo root: e:\eve-rag
src exists: True
✓ src.utils.paths import
PATH_ROOT: E:\eve-rag


## 6. Processed manifests


In [39]:
manifest_files = [
    TRAIN_MANIFEST,
    VAL_MANIFEST,
    TEST_MANIFEST,
    VERITE_TEST_MANIFEST,
    MMFAKEBENCH_VAL_MANIFEST,
    MMFAKEBENCH_TEST_MANIFEST,
    ALL_MANIFEST,
]

for path in manifest_files:
    check(path.name, path.exists(), str(path.relative_to(ROOT)) if path.exists() else "missing")

if ALL_MANIFEST.exists():
    import pandas as pd
    all_df = pd.read_csv(ALL_MANIFEST, dtype={"image_path": "string", "image_id": "string"}, low_memory=False)
    print("\nall.csv rows:", len(all_df))
    print("rows with local images:", int(all_df["has_image"].sum()))
    print("datasets:", all_df["dataset"].value_counts().to_dict())
    print("labels:", all_df["label"].value_counts(dropna=False).to_dict())


✓ train.csv — data\processed\ooc_pairs\manifests\train.csv
✓ val.csv — data\processed\ooc_pairs\manifests\val.csv
✓ test.csv — data\processed\ooc_pairs\manifests\test.csv
✓ verite_test.csv — data\processed\ooc_pairs\manifests\verite_test.csv
✓ mmfakebench_val.csv — data\processed\ooc_pairs\manifests\mmfakebench_val.csv
✓ mmfakebench_test.csv — data\processed\ooc_pairs\manifests\mmfakebench_test.csv
✓ all.csv — data\processed\ooc_pairs\manifests\all.csv

all.csv rows: 97361
rows with local images: 11621
datasets: {'newsclippings': 85360, 'mmfakebench': 11000, 'verite': 1001}
labels: {1: 51043, 0: 46318}


## 7. Flat processed image folder


In [40]:
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".webp", ".bmp", ".tif", ".tiff", ".gif"}

check("processed image folder", PROCESSED_IMAGES.exists(), str(PROCESSED_IMAGES.relative_to(ROOT)))
if PROCESSED_IMAGES.exists():
    flat_images = [p for p in PROCESSED_IMAGES.iterdir() if p.is_file() and p.suffix.lower() in IMAGE_EXTS]
    nested_dirs = [p for p in PROCESSED_IMAGES.iterdir() if p.is_dir()]
    print("flat image files:", len(flat_images))
    check("no nested folders inside processed images", len(nested_dirs) == 0, f"nested folders: {[p.name for p in nested_dirs[:10]]}")

if ALL_MANIFEST.exists():
    import pandas as pd
    df = pd.read_csv(ALL_MANIFEST, dtype={"image_path":"string"}, low_memory=False)
    broken = df[(df["has_image"] == 1) & (~df["image_path"].fillna("").map(lambda p: (ROOT / p).exists()))]
    check("manifest image paths", len(broken) == 0, f"broken={len(broken)}")


✓ processed image folder — data\processed\ooc_pairs\images
flat image files: 11495
✓ no nested folders inside processed images — nested folders: []
✓ manifest image paths — broken=0


## 8. Split files


In [41]:
expected_splits = [
    "train_pair_ids.txt",
    "val_pair_ids.txt",
    "test_pair_ids.txt",
    "verite_test_pair_ids.txt",
    "mmfakebench_val_pair_ids.txt",
    "mmfakebench_test_pair_ids.txt",
    "all_pair_ids.txt",
    "local_image_pair_ids.txt",
]

check("data/splits folder", SPLITS.exists(), str(SPLITS.relative_to(ROOT)))
for name in expected_splits:
    path = SPLITS / name
    check(name, path.exists(), str(path.relative_to(ROOT)) if path.exists() else "missing")


✓ data/splits folder — data\splits
✓ train_pair_ids.txt — data\splits\train_pair_ids.txt
✓ val_pair_ids.txt — data\splits\val_pair_ids.txt
✓ test_pair_ids.txt — data\splits\test_pair_ids.txt
✓ verite_test_pair_ids.txt — data\splits\verite_test_pair_ids.txt
✓ mmfakebench_val_pair_ids.txt — data\splits\mmfakebench_val_pair_ids.txt
✓ mmfakebench_test_pair_ids.txt — data\splits\mmfakebench_test_pair_ids.txt
✓ all_pair_ids.txt — data\splits\all_pair_ids.txt
✓ local_image_pair_ids.txt — data\splits\local_image_pair_ids.txt


## 9. Paper/cache folders


In [42]:
for label, path in [
    ("cache", CACHE),
    ("stream_a cache", STREAM_A_CACHE),
    ("stream_b cache", STREAM_B_CACHE),
    ("stream_c cache", STREAM_C_CACHE),
    ("fusion cache", FUSION_CACHE),
    ("paper", PAPER),
    ("figures", FIGURES),
    ("tables", TABLES),
]:
    path.mkdir(parents=True, exist_ok=True)
    check(label, path.exists(), str(path.relative_to(ROOT)))


✓ cache — cache
✓ stream_a cache — cache\stream_a
✓ stream_b cache — cache\stream_b
✓ stream_c cache — cache\stream_c
✓ fusion cache — cache\fusion
✓ paper — paper
✓ figures — paper\figures
✓ tables — paper\tables


## 10. Final recommendation

If any processed manifest or split file is missing, run `data_setup_updated.ipynb` next. If everything is green, run `eda_updated.ipynb`.
